In [ ]:
#@title Prevent disconnections
%%html
<audio src="https://oobabooga.github.io/silence.m4a" controls>

In [ ]:
import os

SWARMPATH = '/content/'

# Install dotnet dependencies
!wget https://dot.net/v1/dotnet-install.sh -O dotnet-install.sh
!chmod +x dotnet-install.sh
!./dotnet-install.sh --channel 8.0

# Install Clouldflared
!wget https://github.com/cloudflare/cloudflared/releases/download/2024.8.2/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

# use the chosen path
os.environ['SWARMPATH'] = SWARMPATH
%cd $SWARMPATH

# Colab breaks venv, so, tell swarm to not make a venv
os.environ['SWARM_NO_VENV'] = 'true'

# Download SwarmUI
url = "https://github.com/mcmonkeyprojects/SwarmUI"
!git clone $url

In [ ]:
#@title Select & Download Model
KEY = ""  # @param {type:"string"}
KEY = "token=" + KEY.strip()

MODELS = {
    "deepDarkHentaiMixNSFW_v61Hybrid": {
        "file": "deepDarkHentaiMixNSFW_v61Hybrid.safetensors",
        "url": f"https://civitai.com/api/download/models/634653?type=Model&format=SafeTensor&size=pruned&fp=fp16&{KEY}"
    },
    "cyberrealisticPony_semiRealV40": {
        "file": "cyberrealisticPony_semiRealV40.safetensors",
        "url": f"https://civitai.com/api/download/models/2268768?type=Model&format=SafeTensor&size=pruned&fp=fp16&{KEY}"
    },
    "novaCartoon_v10": {
        "file": "novaCartoon_v10.safetensors",
        "url": f"https://civitai.com/api/download/models/821389?type=Model&format=SafeTensor&size=pruned&fp=fp16&{KEY}"
    },
}

CHOICE = "cyberrealisticPony_semiRealV40"  # @param ["cyberrealisticPony_semiRealV40","deepDarkHentaiMixNSFW_v61Hybrid","novaCartoon_v10"]

MODEL = MODELS[CHOICE]
MODEL_PATH = "/content/SwarmUI/Models/checkpoints"
USER_AGENT = '"User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64)"'

!apt install -y aria2 > /dev/null
!mkdir -p $MODEL_PATH

!aria2c --enable-http-keep-alive=false --header=$USER_AGENT --console-log-level=error -c -x 16 -s 16 -k 1M --summary-interval=5 -d $MODEL_PATH -o {MODEL['file']} "{MODEL['url']}"

In [ ]:
# Alright, launch it! Watch the output for a Cloudflare URL
%cd $SWARMPATH/SwarmUI/

# (Colab stupid-proofing: aggressive git ultraforce pull)
!git fetch
!git reset --hard origin/master
!git pull --autostash
# (Colab stupid-proofing: drive wonks the perms so this needs an aggressive rebuild)
!rm -rf ./src/bin/live_release

# Force dotnet 8: disable the script's dotnet 10 auto-install (CI builds/runs master on 8.x)
logic = 'launchtools/linux-build-logic.sh'
cond = 'if [ "$DOTNET_ROOT" = "$SCRIPT_DIR/.dotnet" ] || [ "$DOTNET_ROOT" = "$HOME/.dotnet" ]; then'
with open(logic) as f:
    content = f.read()
if cond in content:
    with open(logic, 'w') as f:
        f.write(content.replace(cond, 'if false; then  # dotnet 8 forced'))
    print('Patched: dotnet 10 auto-install disabled')
else:
    print('Patch not needed')

!bash ./launch-linux.sh --launch_mode none --cloudflared-path cloudflared